In [2]:
import sys
import os
import math
import numpy as np

states = { "s": 0, "E": 1, "5": 2, "I" : 3, "e": 4}
id2state = {0: "s", 1: "E", 2: "5", 3: "I", 4: "e"}

state_transition_prob = np.array([[0.0, 1.0, 0.0, 0.0, 0.0],
                                  [0.0, 0.9, 0.1, 0.0, 0.0],
                                  [0.0, 0.0, 0.0, 1.0, 0.0],
                                  [0.0, 0.0, 0.0, 0.9, 0.1],
                                  [0.0, 0.0, 0.0, 0.0, 0.0]])
emission_nuc_codes = {'A': 0,
                      'C': 1,
                      'G': 2,
                      'T': 3}

emission_probs = np.array([[0.00, 0.00, 0.00, 0.00],
                           [0.25, 0.25, 0.25, 0.25],
                           [0.05, 0.00, 0.95, 0.00],
                           [0.40, 0.10, 0.10, 0.40],
                           [0.00, 0.00, 0.00, 0.00]])

query_sequence = "CTTCATGTGAAAGCAGACGTAAGTCA"


In [3]:
def get_log_prob_for_state_path (state_path, query_sequence):
    res = math.log(0.25)
    for i in range(1, len(state_path)):
        res += math.log(state_transition_prob[ states[state_path[i-1]] ][ states[state_path[i]] ]*emission_probs[ states[state_path[i]] ][ emission_nuc_codes[query_sequence[i]] ])
    return res

In [4]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEE5IIIIIIIIIIIIIIIIIII
k1 = get_log_prob_for_state_path("EEEEEE5IIIIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") +  math.log (0.1)
print (k1)


-43.89740030179307


In [5]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEE5IIIIIIIIIIIIIIIII
k2 = get_log_prob_for_state_path("EEEEEEEE5IIIIIIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k2)


-43.45111319916465


In [6]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEE5IIIIIIIIIIIII
k3 = get_log_prob_for_state_path("EEEEEEEEEEEE5IIIIIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k3)


-43.944833355027704


In [7]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEE5IIIIIIIIII
k4 = get_log_prob_for_state_path("EEEEEEEEEEEEEEE5IIIIIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k4)


-42.58225552052512


In [8]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEE5IIIIIII
k5 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEE5IIIIIII", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k5)


-41.21967768602254


In [9]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEEEEEE5III
k6 = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEE5III", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (k6)


-41.713397841885595


In [10]:
# CTTCATGTGAAAGCAGACGTAAGTCA
# EEEEEEEEEEEEEEEEEEEEEEEEEE
only_E = get_log_prob_for_state_path("EEEEEEEEEEEEEEEEEEEEEEEEEE", "CTTCATGTGAAAGCAGACGTAAGTCA") + math.log (0.1)
print (only_E)


-40.98025137355685


### Design of the Viterbi Value matrix

Rows correspond to the hidden states, and the columns correspond to the emissions that is the observed nucleotide sequences. Here I am showing the calculation for the first two nucletides.

```
             C                                                          T     T
s [s-s-C(0.00) max(s-s-C-s-T, s-E-C-s-T, s-5-C-s-T, s-I-C-s-T, s-e-C-s-T)     .]
E [s-E-C(0.25) max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)     .]
5 [s-5-C(0.00) max(s-s-C-5-T, s-E-C-5-T, s-5-C-5-T, s-I-C-5-T, s-e-C-5-T)     .]
I [s-I-C(0.00) max(s-s-C-I-T, s-E-C-I-T, s-5-C-I-T, s-I-C-I-T  s-e-C-I-T)     .]
e [s-e-C(0.00) max(s-s-C-e-T, s-E-C-e-T, s-5-C-e-T, s-I-C-e-T, s-e-C-e-T)     .]

```

It is important to remember that you will be working in the log scale.

In [35]:
# Initiate two matrices:
# viterbi_value_matrix: to store the values described in the documentation above
# viterbi_trace_matrix: to store the path the lead to the the maximum value in each cell
# For example, the first column of viterbi_trace_matrix will be
# [0] indicating start state released `C`: even though not possible - but we just initiate
# [1] indicating Exon state released `C`:
# [2] indicating 5'ss state released `C`: even though not possible - but we just initiate
# [3] indicating Intron state released `C`:
# [4] indicating end state released `C`: even though not possible - but we just initiate

# Initiate viterbi matrices
n = len(states)
l = len(query_sequence)

# stores maximum log probabilities
val_mat = np.full((n, l),float('-inf'))

# stores traceback path
tr_mat = np.zeros((n, l),dtype=int)

# Initialize first column
first= query_sequence[0]

# initialize other states
for current_state in range(n):
    trans_prob = state_transition_prob[states["s"]][current_state]
    em_prob = emission_probs[current_state][emission_nuc_codes[first]]
    if trans_prob == 0 or em_prob == 0:
        val_mat[current_state][0] = float('-inf')
    else:
        val_mat[current_state][0] = (math.log(trans_prob)+math.log(em_prob))

    tr_mat[current_state][0] = states["s"]

### Implementation of Viterbi Algorithm
Write a function `calculate_prob_for_a_node()` that populate a single cell in the matrix. The function will return two values:
1. the maximum value, for example, look at the 2nd row, 2nd column in the matrix: `max(s-s-C-E-T, s-E-C-E-T, s-5-C-E-T, s-I-C-E-T, s-e-C-E-T)`. If the probability for `s-E-C-E-T` is highest (lets say X), then the function should return `X`

**AND**

2. The index of that maximum value described in the first point: so index of X is `1` (recall that Python works on the 0-based index system)

- Populate `viterbi_value_matrix` with `X` for row 2 and col 2

- Populate `viterbi_trace_matrix` with `1` for row 2 and col 2

In [31]:
# Write for loops to iterate over the whole Viterbi Value matrix.
# Each time, call the function

# function to calculate single node

def calculate_prob_for_a_node(cur_state,cur_col):
    nucleotide = query_sequence[cur_col]
    mx_val = float('-inf')
    mx_idx = 0

    # try all previous states
    for prev in range(n):
        prev_val = val_mat[prev][cur_col - 1]
        trans_prob = state_transition_prob[prev][cur_state]
        em_prob = emission_probs[cur_state][emission_nuc_codes[nucleotide]]

        # skip impossible paths
        if (prev_val == float('-inf') or trans_prob == 0 or em_prob == 0):
            continue
        cur_val = (prev_val + math.log(trans_prob) + math.log(em_prob))

        # update maximum
        if cur_val > mx_val:
            mx_val = cur_val
            mx_idx = prev
    return mx_val, mx_idx


# fill the matrices
for column in range(1, l):
    for cur in range(n):
        mx_val, mx_idx = calculate_prob_for_a_node(cur,column)
        val_mat[cur][column] = mx_val
        tr_mat[cur][column] = mx_idx
print(val_mat)
print()
print(tr_mat)

[[        -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf         -inf         -inf         -inf         -inf
          -inf]
 [ -1.38629436  -2.87794924  -4.36960411  -5.86125899  -7.35291387
   -8.84456875 -10.33622362 -11.8278785  -13.31953338 -14.81118825
  -16.30284313 -17.79449801 -19.28615288 -20.77780776 -22.26946264
  -23.76111751 -25.25277239 -26.74442727 -28.23608214 -29.72773702
  -31.2193919  -32.71104677 -34.20270165 -35.69435653 -37.1860114
  -38.67766628]
 [        -inf         -inf         -inf         -inf -11.15957636
          -inf -11.19844713         -inf -14.18175689 -18.61785074
  -20.10950562 -21.6011605  -20.14837639         -inf -26.07612513
  -24.62334102 -29.05943488         -inf -29.09830565         -inf
  -35.02605439 -36.51770926 -35

In [34]:
# Write a function to trace the state path that gave the maximum probability.
# This will be the final result.


# HINT: You should first find the maximum value in the last column of `val_mat`,
# because that is the one with the largest probability.
# The index of that value is the state of the last nucleotide.

def trace_state_path():
    # transition to end state
    end = states["e"]
    scores = np.full(n, float('-inf'))
    for state in range(n):
        trans_prob = state_transition_prob[state][end]

        if (trans_prob > 0 and val_mat[state][-1] != float('-inf')):
            scores[state] = (val_mat[state][-1]+ math.log(trans_prob))

    best_final_state = np.argmax(scores)
    best_prob = scores[best_final_state]

    # traceback
    path = [best_final_state]

    for column in range(l - 1, 0, -1):
        path.append(tr_mat[path[-1]][column])
    path.reverse()

    # convert ids to states
    best_path = ""
    for state_id in path:
        if id2state[state_id] != "s":
            best_path += id2state[state_id]
    return best_prob, best_path


best_probability, best_path = trace_state_path()

print("Best Log Probability:")
print(best_probability)

print()

print("Best Hidden State Path:")
print(best_path)

Best Log Probability:
-41.21967768602254

Best Hidden State Path:
EEEEEEEEEEEEEEEEEE5IIIIIII
